# VAE V2 — KL Warm-up Training and Final Comparison
This notebook preserves V1 and trains V2 with linear beta annealing for epochs 1–10.

In [1]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


In [2]:
!pip install -q kagglehub scikit-image torchmetrics torch-fidelity
from pathlib import Path
import os, subprocess, sys
PROJECT_ROOT = Path('/content/Digital-Evidence-GenAI')
if not PROJECT_ROOT.exists():
    subprocess.run(['git','clone','https://github.com/chetanraje27/Digital-Evidence-GenAI.git',str(PROJECT_ROOT)], check=True)
sys.path.insert(0, str(PROJECT_ROOT/'src'))
print('Project:', PROJECT_ROOT)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 7.1 MB/s eta 0:00:00
Project: /content/Digital-Evidence-GenAI


In [3]:
# Download CASIA only when the manifest target is unavailable.
import csv, kagglehub
first = next(csv.DictReader(open(PROJECT_ROOT/'data/splits/train.csv', encoding='utf-8')))
expected = PROJECT_ROOT / first['image_path']
if not expected.exists():
    downloaded = Path(kagglehub.dataset_download('divg07/casia-20-image-tampering-detection-dataset'))
    casia2 = next(downloaded.rglob('CASIA2'))
    target = PROJECT_ROOT/'data/raw/CASIA2'
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists(): os.symlink(casia2, target, target_is_directory=True)
print('Dataset ready:', expected.exists())

100%|██████████| 2.56G/2.56G [02:14<00:00, 20.3MB/s]

Extracting files...


Dataset ready: True


In [4]:
# Full V2 training: writes only V2 paths and never overwrites V1.
from argparse import Namespace
from train_vae_v2 import train
args = Namespace(
    splits_dir=PROJECT_ROOT/'data/splits', checkpoint_path=PROJECT_ROOT/'checkpoints/best_vae_v2.pth',
    history_path=PROJECT_ROOT/'results/vae_v2_training_history.csv',
    curve_path=PROJECT_ROOT/'outputs/vae/vae_v2_loss_curves.png',
    summary_path=PROJECT_ROOT/'results/vae_v2_training_summary.json',
    image_size=128, batch_size=32, num_workers=2, latent_dim=128,
    learning_rate=0.0005, max_epochs=50, patience=5, target_beta=0.001,
    warmup_epochs=10, seed=42, smoke_test=False, smoke_batches=2)
training_summary = train(args)
training_summary

Epoch 01/50 | beta=0.0001000 | train_recon=0.04381708 | train_kl=27.542223 | train_total=0.04657130 | val_recon=0.03252583 | val_kl=40.088727 | val_total=0.03653471 | seconds=29.0
Epoch 02/50 | beta=0.0002000 | train_recon=0.03352262 | train_kl=25.578313 | train_total=0.03863828 | val_recon=0.03236278 | val_kl=27.515644 | val_total=0.03786590 | seconds=28.4
Epoch 03/50 | beta=0.0003000 | train_recon=0.03268875 | train_kl=19.252747 | train_total=0.03846458 | val_recon=0.03208854 | val_kl=19.048424 | val_total=0.03780307 | seconds=27.4
Epoch 04/50 | beta=0.0004000 | train_recon=0.03255141 | train_kl=15.168056 | train_total=0.03861863 | val_recon=0.03205166 | val_kl=15.026757 | val_total=0.03806236 | seconds=28.2
Epoch 05/50 | beta=0.0005000 | train_recon=0.03246524 | train_kl=13.104804 | train_total=0.03901764 | val_recon=0.03187903 | val_kl=13.024048 | val_total=0.03839105 | seconds=27.8
Epoch 06/50 | beta=0.0006000 | train_recon=0.03176026 | train_kl=12.154654 | train_total=0.03905306 

{'device': 'Tesla T4',
 'epochs_completed': 48,
 'best_epoch': 43,
 'best_validation_total_loss': 0.04100875587972476,
 'early_stopping_triggered': True,
 'training_time_seconds': 1363.9269068180001,
 'checkpoint_path': '/content/Digital-Evidence-GenAI/checkpoints/best_vae_v2.pth',
 'history_path': '/content/Digital-Evidence-GenAI/results/vae_v2_training_history.csv',
 'target_beta': 0.001,
 'warmup_epochs': 10,
 'smoke_test': False}

In [5]:
# Complete V2 test evaluation and standard 2048-feature FID.
from evaluate_vae_v2 import evaluate
eval_args = Namespace(
    splits_dir=PROJECT_ROOT/'data/splits', v1_checkpoint=PROJECT_ROOT/'checkpoints/best_vae.pth',
    v2_checkpoint=PROJECT_ROOT/'checkpoints/best_vae_v2.pth',
    v1_metrics=PROJECT_ROOT/'results/vae_test_metrics.json',
    metrics_path=PROJECT_ROOT/'results/vae_v2_test_metrics.json',
    per_image_path=PROJECT_ROOT/'results/vae_v2_test_per_image_metrics.csv',
    comparison_path=PROJECT_ROOT/'results/vae_v1_vs_v2_comparison.csv',
    grid_path=PROJECT_ROOT/'outputs/vae/vae_v1_vs_v2_reconstruction.png',
    image_size=128, batch_size=32, num_workers=2, latent_dim=128, beta=0.001,
    seed=42, expected_test_count=1892, grid_per_class=3)
comparison = evaluate(eval_args)
comparison

Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth
100%|██████████| 91.2M/91.2M [00:02<00:00, 42.7MB/s]


{'v1': {'version': 'V1 baseline',
  'mse': 0.029098848117974214,
  'psnr': 15.789472667661832,
  'ssim': 0.287553520353381,
  'fid': 339.5789794921875},
 'v2': {'version': 'V2 KL warm-up',
  'mse': 0.028168018482011167,
  'psnr': 15.934777124235545,
  'ssim': 0.2901890495295851,
  'fid': 321.3583068847656,
  'kl': 8.219713989322331},
 'evaluation_seconds': 42.826706351999974}

In [6]:
v1, v2 = comparison['v1'], comparison['v2']
print('V1:')
print('MSE', v1['mse']); print('PSNR', v1['psnr']); print('SSIM', v1['ssim']); print('FID', v1['fid'])
print('\nV2:')
print('MSE', v2['mse']); print('PSNR', v2['psnr']); print('SSIM', v2['ssim']); print('FID', v2['fid']); print('KL', v2['kl'])
print('\nBest epoch', training_summary['best_epoch'])
print('Training time', training_summary['training_time_seconds'])
recon_improved = v2['mse'] < v1['mse'] and v2['psnr'] > v1['psnr'] and v2['ssim'] > v1['ssim']
generation_improved = v2['fid'] < v1['fid']
print('V2 improved reconstruction:', recon_improved)
print('V2 improved generation:', generation_improved)

V1:
MSE 0.029098848117974214
PSNR 15.789472667661832
SSIM 0.287553520353381
FID 339.5789794921875

V2:
MSE 0.028168018482011167
PSNR 15.934777124235545
SSIM 0.2901890495295851
FID 321.3583068847656
KL 8.219713989322331

Best epoch 43
Training time 1363.9269068180001
V2 improved reconstruction: True
V2 improved generation: True
